# IEAP – Python series 03: Assignment - Group Lagarde | Long | Deemu

| Part | Student | Sections | Branch |
|---|---|---|---|
|  **A** | **LAGARDE Alexis** (Student A) | Section 2 - find and plot remarkable points (2.1, 2.2, 2.3) + common setup | `part-a` |
|  **B** | **LONG Jiangbin** (Student B) | Section 3 - analyze a known signal (3.1, 3.2, 3.3) | `part-b` |
|  **C** | **DEEMU Charles Sam** (Student C) | Section 4 - noisy signal and low-pass filter (4.1, 4.2, 4.3) | `part-c` |

Working on different files were thought, but as for part B we need function from part A and same for part C, we decided to work on the same file with for every part a new branch based on a up to date `main` branch

# PART A - LAGARDE Alexis
**Section 2 – functions to find and plot remarkable points (2.1, 2.2, 2.3)**

### Imports
Legends aren't displaying correctly if `matplotlib widget` option is uncommented, clear all outputs and restart the kernel if plots are still zoomable (used for interactive plots in the notebook)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
# %matplotlib widget   

## 2. Create functions to find and plot remarkable points

Reference signal for the tests:

In [ ]:
index = np.arange(10)
signal = np.array([-1, 1, 2, 1, 0, -1, 1, 2, 1, -2])
print("  index = ", index.tolist())
print(" signal = ", signal.tolist())

### 2.1. Functions to find zero crossings and to plot them

The functions below are those of the live-coding session, **with the docstrings and comments improved as asked in 2.2**.

**Definition used:** a crossing is reported at index `i` when the sign changes between the last non-zero sample
at (or before) `i` and sample `i + 1`. Samples equal to 0 have no sign of their own: they inherit the sign of the
previous non-zero sample (leading zeros take the sign of the first non-zero sample).
So `[-1, 0, 0, 1]` has one positive crossing at index 2, `[1, 0, 1]` has none, and an all-zero signal has none.

In [5]:
def find_zero_crossings_no_zeros(signs: np.ndarray):
    """Find positive and negative zero crossings in a sign array WITHOUT zeros.

    Parameters
    ----------
    signs : np.ndarray of int, shape (n,)
        Sign of each sample, containing only -1 and +1 (no 0). Use
        :func:`propagate_signs_over_zeros` first if the array may contain 0.

    Returns
    -------
    i_cross_pos : np.ndarray of int
        Indices ``i`` such that ``signs[i] == -1`` and ``signs[i + 1] == +1``
        (the signal goes from negative to positive between ``i`` and ``i + 1``).
    i_cross_neg : np.ndarray of int
        Indices ``i`` such that ``signs[i] == +1`` and ``signs[i + 1] == -1``.
    """
    # diff[i] = signs[i+1] - signs[i] is +2 on a -1 -> +1 change,
    # -2 on a +1 -> -1 change, and 0 when the sign stays the same.
    diff = np.diff(signs)
    # np.where(...) returns a tuple (one array per dimension): take [0] for 1D
    i_cross_pos = np.where(diff > 0)[0]
    i_cross_neg = np.where(diff < 0)[0]
    return i_cross_pos, i_cross_neg

def propagate_signs_over_zeros(signs: np.ndarray):
    """Replace every 0 of a sign array by the nearest meaningful non-zero sign.

    * a 0 takes the sign of the closest non-zero sample **before** it
      (forward fill);
    * leading zeros (no non-zero sample before them) take the sign of the
      first non-zero sample (backward fill);
    * an array made only of zeros is returned unchanged.

    Parameters
    ----------
    signs : np.ndarray of int, shape (n,)
        Array of -1, 0, +1 (typically ``np.sign(s)``). It is not modified.

    Returns
    -------
    signs_no_zeros : np.ndarray of int, shape (n,)
        Copy of ``signs`` where zeros have been replaced as described above.

    Example
    -------
    >>> propagate_signs_over_zeros(np.array([0, 0, 1, 0, -1, 0]))
    array([ 1,  1,  1,  1, -1, -1])
    """
    signs_no_zeros = signs.copy()  # work on a copy: never modify the input

    # Forward fill: each zero gets the sign of the previous non-zero
    mask = signs_no_zeros != 0  # True where the sample has a sign of its own
    # For each position, keep its own index if it is non-zero, else 0.
    # e.g. signs = [ 1, 0, 0,-1, 0] -> idx = [0, 0, 0, 3, 0]
    idx = np.where(mask, np.arange(len(signs_no_zeros)), 0)
    # Running maximum: each position now holds the index of the LAST non-zero
    # sample at or before it.  idx = [0, 0, 0, 3, 3]
    idx = np.maximum.accumulate(idx)
    # Fancy indexing: copy the sign found at that index. -> [1, 1, 1, -1, -1]
    signs_no_zeros = signs_no_zeros[idx]

    # Backward fill of leading zeros
    # If the array starts with zeros, step 1 pointed them to index 0, which is
    # itself 0, so they are still 0. Give them the sign of the first non-zero.
    if signs_no_zeros[0] == 0:
        i_first_nonzero = np.where(signs != 0)[0]
        if len(i_first_nonzero) > 0:  # otherwise the array is all zeros
            signs_no_zeros[: i_first_nonzero[0]] = signs[i_first_nonzero[0]]

    return signs_no_zeros

def find_zero_crossings(s: np.ndarray):
    """Find the positive and negative zero crossings of a 1D signal.

    Parameters
    ----------
    s : array_like, shape (n,)
        The signal (list or numpy array, int or float).

    Returns
    -------
    i_cross_pos : np.ndarray of int
        Indices ``i`` where the signal goes from negative to positive between
        ``i`` and ``i + 1`` (rising edge, positive slope).
    i_cross_neg : np.ndarray of int
        Indices ``i`` where the signal goes from positive to negative between
        ``i`` and ``i + 1`` (falling edge, negative slope).

    Raises
    ------
    ValueError
        If ``s`` is not one-dimensional.

    Notes
    -----
    Samples equal to 0 are handled by :func:`propagate_signs_over_zeros`
    (see the module docstring for the exact definition).
    """
    # Input validation
    s = np.asarray(s)  # accept lists / tuples as well as arrays
    if s.ndim != 1:
        raise ValueError("Input must be a 1D array")
    if s.size < 2:  # a crossing needs at least two samples
        return np.array([], dtype=int), np.array([], dtype=int)

    # Keep only the sign of each sample: -1, 0 or +1 
    signs = np.sign(s).astype(int)

    # Remove zeros only if there are some (saves work otherwise) 
    if np.any(signs == 0):
        signs_no_zeros = propagate_signs_over_zeros(signs)
    else:
        signs_no_zeros = signs

    # A crossing is now simply a change of sign between neighbours 
    return find_zero_crossings_no_zeros(signs_no_zeros)

Plotting functions (`plot_` = no `plt.show()`, `display_` = shows):

In [ ]:
def plot_signal(i: np.ndarray, s: np.ndarray, color: str = "blue",
                label: str = "signal", xlabel: str = "index",
                ylabel: str = "signal"):
    """Plot a signal as dots joined by a thin line, with a horizontal zero line.

    Parameters
    ----------
    i : np.ndarray, shape (n,)
        x values (sample indices or time vector).
    s : np.ndarray, shape (n,)
        Signal values.
    color : str
        Matplotlib color of the signal.
    label : str
        Legend label.
    xlabel, ylabel : str
        Axis labels.

    Returns
    -------
    None. Draws on the current axes, does not show the figure.
    """
    plt.axhline(y=0, color="gray", linestyle="-")
    plt.plot(i, s, ".-", markersize=5, linewidth=0.25, color=color, label=label)
    plt.ylabel(ylabel)
    plt.xlabel(xlabel)

def plot_remarkable_points(t: np.ndarray, s: np.ndarray, format: str, label: str):
    """Plot a set of remarkable points with big markers.

    Parameters
    ----------
    t : np.ndarray
        x values of the points only (e.g. ``time[i_points]``).
    s : np.ndarray
        y values of the points only (e.g. ``signal[i_points]``).
    format : str
        Matplotlib format string, e.g. ``"^r"`` (red up-triangles).
    label : str
        Legend label.

    Returns
    -------
    None. Draws on the current axes, does not show the figure.
    """
    plt.plot(t, s, format, markersize=10, label=label)

def plot_positive_zero_crossings(i: np.ndarray, s: np.ndarray):
    """Find and plot the positive zero crossings of ``s`` (red up-triangles).

    Parameters
    ----------
    i : np.ndarray, shape (n,)   x values (indices or time).
    s : np.ndarray, shape (n,)   signal.
    """
    i_pos, _ = find_zero_crossings(s)
    plot_remarkable_points(i[i_pos], s[i_pos], "^r", "positive zero crossings")

def plot_negative_zero_crossings(i: np.ndarray, s: np.ndarray):
    """Find and plot the negative zero crossings of ``s`` (yellow down-triangles).

    Parameters
    ----------
    i : np.ndarray, shape (n,)   x values (indices or time).
    s : np.ndarray, shape (n,)   signal.
    """
    _, i_neg = find_zero_crossings(s)
    plot_remarkable_points(i[i_neg], s[i_neg], "vy", "negative zero crossings")

def plot_zero_crossings(i: np.ndarray, s: np.ndarray):
    """Find and plot both positive and negative zero crossings of ``s``."""
    plot_positive_zero_crossings(i, s)
    plot_negative_zero_crossings(i, s)

def finalize_plot():
    """Add the legend (outside, on the right) and the grid, then show.

    NOTE: a legend outside the axes does not work well with
    ``%matplotlib widget``.
    """
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True)
    plt.show()

def display_signal_and_crossings(i: np.ndarray, s: np.ndarray):
    """Show a new figure with the signal and its zero crossings.

    Parameters
    ----------
    i : np.ndarray, shape (n,)   x values (indices or time).
    s : np.ndarray, shape (n,)   signal.
    """
    plt.figure()
    plot_signal(i, s)
    plot_zero_crossings(i, s)
    finalize_plot()

In [ ]:
display_signal_and_crossings(index, signal)
i_cross_pos, i_cross_neg = find_zero_crossings(signal)
print("positive zero crossings:", i_cross_pos, "  negative zero crossings:", i_cross_neg)